In [1]:
# ==========================================================
# Experiment 10
# Fine-Tuning DistilBERT for Sentiment Classification
# ==========================================================

# Install Required Libraries
!pip install -q transformers==4.41.2 datasets torch

# ----------------------------------------------------------
# Import Libraries
# ----------------------------------------------------------

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# ----------------------------------------------------------
# Sample Dataset
# ----------------------------------------------------------

texts = [
    "This product is excellent.",
    "I really enjoyed this movie.",
    "The service was terrible.",
    "The food was awful.",
    "Amazing experience.",
    "Very disappointing."
]

labels = [1, 1, 0, 0, 1, 0]

# ----------------------------------------------------------
# Load Tokenizer
# ----------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# ----------------------------------------------------------
# Create Dataset
# ----------------------------------------------------------

class SentimentDataset(Dataset):

    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=64
        )
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

dataset = SentimentDataset(texts, labels)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

# ----------------------------------------------------------
# Load DistilBERT Model
# ----------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)

# ----------------------------------------------------------
# Optimizer
# ----------------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

# ----------------------------------------------------------
# Training
# ----------------------------------------------------------

epochs = 3

model.train()

for epoch in range(epochs):

    total_loss = 0

    for batch in loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# ----------------------------------------------------------
# Evaluation
# ----------------------------------------------------------

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("\nAccuracy:", round(accuracy * 100, 2), "%")

# ----------------------------------------------------------
# Test Prediction
# ----------------------------------------------------------

test_text = "The movie was fantastic."

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    padding=True
).to(device)

with torch.no_grad():

    output = model(**inputs)

prediction = torch.argmax(output.logits).item()

if prediction == 1:
    print("\nPrediction: Positive")
else:
    print("\nPrediction: Negative")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss: 2.1367
Epoch 2 Loss: 1.9740
Epoch 3 Loss: 1.9668

Accuracy: 100.0 %

Prediction: Positive
